In [ ]:
from dotenv import load_dotenv, find_dotenv
assert load_dotenv(find_dotenv(usecwd=False)), "The .env file was not loaded."

import pickle
from pathlib import Path


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from drn import *
import seaborn as sns

from generate_synthetic_dataset import generate_synthetic_gamma_lognormal

torch.set_num_threads(1)

In [ ]:
from train_new import train
from glm_new import *

In [ ]:
features, target, means, dispersion = generate_synthetic_gamma_lognormal(20000)

In [ ]:
x_train, x_val, x_test, y_train, y_val, y_test,\
      x_train_raw, x_val_raw, x_test_raw,\
          num_features, cat_features,\
             all_categories, ct =\
                split_and_preprocess(features, target, ['X_1', 'X_2'], [], seed = 42, num_standard = False)
x_train

In [ ]:
# We'll use lowercase "x_train" "y_train" for pandas dataframes
# and uppercase "X_train" "Y_train" for tensors.
X_train = torch.Tensor(x_train.values)
Y_train = torch.Tensor(y_train.values)
X_val = torch.Tensor(x_val.values)
Y_val = torch.Tensor(y_val.values)
X_test = torch.Tensor(x_test.values)
Y_test = torch.Tensor(y_test.values)

train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

# Training

In [ ]:
MODEL_DIR = Path("models/synth")

glm = torch.load(MODEL_DIR / "glm.pkl", weights_only=False)
cann = torch.load(MODEL_DIR / "cann.pkl", weights_only=False)
mdn = torch.load(MODEL_DIR / "mdn.pkl", weights_only=False)
ddr = torch.load(MODEL_DIR / "ddr.pkl", weights_only=False)
drn = torch.load(MODEL_DIR / "drn.pkl", weights_only=False)

In [ ]:
best_drn_paras = [0.001, 0.0, 0.0, 0.0, 0.025, 10]

cutpoints_DRN = drn_cutpoints(c_0 = np.min(Y_train.detach().numpy()) * 1.05 if np.min(Y_train.detach().numpy()) < 0 else 0.0,
                              c_K = np.max(Y_train.detach().numpy()) * 1.05,
                              p = best_drn_paras[-2],
                              y = Y_train.detach().numpy(),
                              min_obs =  best_drn_paras[-1])
print(len(cutpoints_DRN))

# best_drn_paras = [1e-3, 1e-2, 0]
torch.manual_seed(23)
drn_small_kl = DRN(num_features = X_train.shape[1], cutpoints = cutpoints_DRN, glm = glm,\
                    hidden_size=256, num_hidden_layers=3,
                      baseline_start = False,  dropout_rate = 0.1)
if True:  
    torch.manual_seed(23)
    train(
                model=drn_small_kl,
                criterion=lambda pred, y: drn_loss(pred, y, kl_alpha = best_drn_paras[1],#best_drn_paras[2]*10, #2e-4
                                                  mean_alpha =best_drn_paras[2],
                                                    dv_alpha =  best_drn_paras[3],
                                                     kl_direction = 'forwards',
                                                     kind = 'jbce'),   
                # criterion_val=lambda pred, y: drn_loss(pred, y, kind = 'jbce'), 
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                batch_size=256,
                epochs=2000,
                patience=50,
                lr=best_drn_paras[0],
                print_details=True,
                log_interval=1,
    )
    drn_small_kl.eval()


In [ ]:
best_drn_paras = [0.001, 0.05, 0.0, 0.0, 0.025, 10]

cutpoints_DRN = drn_cutpoints(c_0 = np.min(Y_train.detach().numpy()) * 1.05 if np.min(Y_train.detach().numpy()) < 0 else 0.0,
                              c_K = np.max(Y_train.detach().numpy()) * 1.05,
                              p = best_drn_paras[-2],
                              y = Y_train.detach().numpy(),
                              min_obs =  best_drn_paras[-1])
print(len(cutpoints_DRN))

# best_drn_paras = [1e-3, 1e-2, 0]
torch.manual_seed(23)
drn_large_kl = DRN(num_features = X_train.shape[1], cutpoints = cutpoints_DRN, glm = glm,\
                    hidden_size=256, num_hidden_layers=3,
                      baseline_start = False,  dropout_rate = 0.1)
if True:  
    torch.manual_seed(23)
    train(
                model=drn_large_kl,
                criterion=lambda pred, y: drn_loss(pred, y, kl_alpha = best_drn_paras[1],#best_drn_paras[2]*10, #2e-4
                                                  mean_alpha =best_drn_paras[2],
                                                    dv_alpha =  best_drn_paras[3],
                                                     kl_direction = 'forwards',
                                                     kind = 'jbce'),   
                # criterion_val=lambda pred, y: drn_loss(pred, y, kind = 'jbce'), 
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                batch_size=256,
                epochs=2000,
                patience=50,
                lr=best_drn_paras[0],
                print_details=True,
                log_interval=1,
    )
    drn_large_kl.eval()


# Visualisatoin

In [ ]:
GRID_SIZE = 3000  # Increase this to get more accurate CRPS estimates
grid = torch.linspace(0.0, np.max(y_train) * 1.1, GRID_SIZE).unsqueeze(-1)
logprob_drn = drn.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
logprob_drn_small_kl = drn_small_kl.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
logprob_drn_large_kl = drn_large_kl.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
logprob_ddr = ddr.distributions(X_test).log_prob(grid).detach().cpu().numpy().T
logprob_baseline = glm.distributions(X_test).log_prob(grid).detach().cpu().numpy().T

In [ ]:

# Convert log-probs to probabilities
prob_drn = np.exp(logprob_drn)
prob_drn_small_kl = np.exp(logprob_drn_small_kl)
prob_drn_large_kl = np.exp(logprob_drn_large_kl)
prob_ddr = np.exp(logprob_ddr)
prob_baseline = np.exp(logprob_baseline)

# Add small epsilon to avoid division by zero or log(0)
epsilon = 1e-12
prob_drn = np.clip(prob_drn, epsilon, None)
prob_drn_small_kl = np.clip(prob_drn_small_kl, epsilon, None)
prob_drn_large_kl = np.clip(prob_drn_large_kl, epsilon, None)
prob_ddr = np.clip(prob_ddr, epsilon, None)

# Compute Δy (uniform grid spacing)
delta_y = (grid[1] - grid[0]).item()

# Compute KL divergence for each test point
kl_divs_drn = np.sum(prob_drn * (np.log(prob_drn) - np.log(prob_baseline)), axis=1) * delta_y
kl_divs_drn_small_kl = np.sum(prob_drn_small_kl * (np.log(prob_drn_small_kl) - np.log(prob_baseline)), axis=1) * delta_y
kl_divs_drn_large_kl = np.sum(prob_drn_large_kl * (np.log(prob_drn_large_kl) - np.log(prob_baseline)), axis=1) * delta_y
kl_divs_ddr = np.sum(prob_ddr * (np.log(prob_ddr) - np.log(prob_baseline)), axis=1) * delta_y
# Define consistent bins for overlapping histogram
bin_width = 0.01/2
max_val = max(np.max(kl_divs_drn), np.max(kl_divs_ddr))
bins = np.arange(0, max_val + bin_width, bin_width)

# Create the plot
plt.figure(figsize=(7, 7))

# Plot KDEs for each KL divergence group
sns.kdeplot(kl_divs_drn_large_kl, label='Model = DRN (KL_Reg = 0.05)', bw_adjust=1.5, linewidth=4, color = 'blue', alpha = 0.25)
sns.kdeplot(kl_divs_drn, label='Model = DRN (KL_Reg = 0.003; Tuned)', bw_adjust=1.5, linewidth=4, color = 'blue', alpha = 1.0)
sns.kdeplot(kl_divs_drn_small_kl, label='Model = DRN (KL_Reg = 0)', bw_adjust=1.5, linewidth=4, color = 'purple', alpha = 1.0)
sns.kdeplot(kl_divs_ddr, label='Model = DDR', bw_adjust=1.5, linewidth=4, color = 'black')
# sns.kdeplot(kl_divs_cann, label='Model = CANN', bw_adjust=1.5, linewidth=2)
# sns.kdeplot(kl_divs_mdn, label='Model = MDN', bw_adjust=1.5, linewidth=2)

# Axes and formatting
plt.title("Distribution of $D_{\\text{KL}}(f_{\\text{Model}}||f_{\\text{GLM}})$ For Different Models", fontsize = 16)
plt.xlabel("$D_{\\text{KL}}(f_{\\text{Model}}||f_{\\text{GLM}})$", fontsize = 14)
plt.ylabel("Empirical Density of $D_{\\text{KL}}(f_{\\text{Model}}||f_{\\text{GLM}})$", fontsize = 14)
plt.xlim([0, 0.2])
# plt.ylim([0, 5000])
plt.legend(fontsize = 14)
plt.tight_layout()
# plt.show()
plt.savefig("plots/synth/_Synthetic_Distr_KL.png");

